# **Setup**

In [1]:
# 1. Install necessary libraries
!pip install torch_geometric sentence-transformers -q

import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool
from torch_geometric.data import Data, Dataset, DataLoader
from sentence_transformers import SentenceTransformer
import json
import os
import numpy as np
from google.colab import drive

# 2. Mount Drive
drive.mount('/content/drive')

# 3. Create a destination folder
!mkdir -p /content/graphs

# 4. Extract the RAR file
# Note the quotes around the path because of the spaces in "group10 social network"
rar_path = "/content/drive/MyDrive/group10 social network/graphs.rar"
!unrar x "{rar_path}" "/content/graphs/"

print(f"Setup Complete. Found {len(os.listdir('/content/graphs'))} graph files.")

Streaming output truncated to the last 5000 lines.
Extracting  /content/graphs/graphs/post_2tmoce.json                       91%  OK 
Extracting  /content/graphs/graphs/post_2tmody.json                       91%  OK 
Extracting  /content/graphs/graphs/post_2tmogu.json                       91%  OK 
Extracting  /content/graphs/graphs/post_2tmok5.json                       91%  OK 
Extracting  /content/graphs/graphs/post_2tmoku.json                       91%  OK 
Extracting  /content/graphs/graphs/post_2tmpav.json                       91%  OK 
Extracting  /content/graphs/graphs/post_2tmpdt.json                       91%  OK 
Extracting  /content/graphs/graphs/post_2tmphe.json                       91%  OK 
Extracting  /content/graphs/graphs/post_2tmpmf.json                       91%  OK 
Extracting  /content/graphs/graphs/post_2tmpmh.json                       91%  OK 
Extracting  /content/graphs/g

# **Load data**

In [5]:
from sentence_transformers import SentenceTransformer
text_model = SentenceTransformer('all-MiniLM-L6-v2')

class RedditDataset(Dataset):
    def __init__(self, root_dir):
        super().__init__()
        self.root_dir = root_dir
        self.files = [f for f in os.listdir(root_dir) if f.endswith('.json')]

    def len(self): return len(self.files)

    def get(self, idx):
        with open(os.path.join(self.root_dir, self.files[idx]), 'r') as f:
            graph_data = json.load(f)

        bodies = [node.get('body', "") for node in graph_data['nodes']]
        text_vecs = text_model.encode(bodies, convert_to_tensor=False, show_progress_bar=False)

        node_features, node_map = [], {}
        for i, node in enumerate(graph_data['nodes']):
            score = np.log1p(max(0, node.get('comment_score', 0)))
            trust = np.log1p(max(0, node.get('author_trust', 0)))

            # Combine the pre-computed vector with metadata
            feat = list(text_vecs[i]) + [score, trust]
            node_features.append(feat)
            node_map[node['id']] = i

        edges = [[node_map[e['source']], node_map[e['target']]]
                 for e in graph_data['edges'] if e['source'] in node_map and e['target'] in node_map]

        x = torch.tensor(node_features, dtype=torch.float)
        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous() if edges else torch.tensor([[0],[0]], dtype=torch.long)

        return Data(x=x, edge_index=edge_index)

# Re-initialize the dataset with the correct path
dataset = RedditDataset('/content/graphs/graphs')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# **GAT Architecture**

In [6]:
class RedditGAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=8):
        super(RedditGAT, self).__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=0.6)
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1, concat=False, dropout=0.6)

        # NEW: The Predictor Layer
        # This turns the 128 vector into a single number (the predicted score)
        self.predictor = torch.nn.Linear(out_channels, 1)

    def forward(self, x, edge_index, batch):
        x = F.dropout(x, p=0.6, training=self.training)
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.6, training=self.training)
        x = self.conv2(x, edge_index)

        # Pool nodes into a single graph representation [Batch, 128]
        graph_vector = global_mean_pool(x, batch)

        # Final prediction [Batch, 1]
        return self.predictor(graph_vector)


# **Training Loop and save model**

In [ ]:
loader = DataLoader(dataset, batch_size=512, shuffle=True, num_workers=0)

device = torch.device('cuda')
model = RedditGAT(in_channels=386, hidden_channels=64, out_channels=128).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)
criterion = torch.nn.MSELoss()

print("Training Started.")

for epoch in range(1, 21): # 20 Epochs is the sweet spot
    model.train()
    total_loss, skipped = 0, 0

    # Simple batch counter so you can see it moving in real-time
    for i, data in enumerate(loader):
        try:
            data = data.to(device)
            optimizer.zero_grad()
            out = model(data.x, data.edge_index, data.batch)
            target = global_mean_pool(data.x[:, 384].view(-1, 1), data.batch)

            loss = criterion(out, target)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

            if i % 10 == 0:
                print(f"Epoch {epoch} | Batch {i}/{len(loader)} processing...", end="\r")

        except Exception:
            skipped += 1
            continue

    avg_loss = total_loss / (len(loader) - skipped)
    print(f"\n Epoch {epoch:03d} | Loss: {avg_loss:.4f} | Skipped: {skipped}")

    # Save the model to your Drive every single epoch
    torch.save(model.state_dict(), '/content/drive/MyDrive/gat_model_ver1.pt')

print("Done!")

/tmp/ipykernel_3096/2572857405.py:1: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  loader = DataLoader(dataset, batch_size=512, shuffle=True, num_workers=0)


Training Started.

 Epoch 001 | Loss: 0.2238 | Skipped: 0

 Epoch 002 | Loss: 0.0915 | Skipped: 0

 Epoch 003 | Loss: 0.0816 | Skipped: 0

 Epoch 004 | Loss: 0.0795 | Skipped: 0

 Epoch 005 | Loss: 0.0765 | Skipped: 0

 Epoch 006 | Loss: 0.0762 | Skipped: 0

 Epoch 007 | Loss: 0.0752 | Skipped: 0

 Epoch 008 | Loss: 0.0735 | Skipped: 0

 Epoch 009 | Loss: 0.0725 | Skipped: 0

 Epoch 010 | Loss: 0.0721 | Skipped: 0
